# Case Study - Data Analysis Airline
Analyst: Mochammad Daffa Putra Karyudi

## Limitation
Meskipun analisa ini didasarkan pada dataset yang kaya, terdapat beberapa keterbatasan fundamental yang harus dipertimbangkan saat mengevaluasi temuan. Keterbatasan utama berasal dari ketiadaan **kamus data (data dictionary)** atau dokumentasi teknis yang menyertainya. Berdasarkan struktur dan nama kolom (misalnya, `pax_name`, `travel_date`, `airline`, `harga`, `class`, `sector`), data ini sangat diindikasikan berasal dari domain industri penerbangan. Namun, tanpa definisi operasional yang formal, analisis bergantung pada asumsi yang berpotensi signifikan, yang termanifestasi dalam keterbatasan berikut:

1.  **Ambiguitas Kritis pada Variabel Kunci (`class`):**
    * **Variabel `class`:** Kolom ini berisi kode-kode seperti **'X', 'Q', 'V', 'L', 'T', 'P'**, dll. Dalam konteks penerbangan, ini kemungkinan besar adalah *Fare Classes* atau *Reservation Booking Designators (RBDs)*. Tanpa metadata, mustahil untuk mengetahui secara pasti:
        * **Hirarki Kabin:** Kelas mana yang termasuk dalam kategori *First Class*, *Business*, *Premium Economy*, atau *Economy*.
        * **Aturan Tarif (Fare Rules):** Apakah sebuah kelas tiket dapat di-refund, diubah jadwalnya, atau berapa banyak bagasi yang diizinkan.
        * **Jenis Tarif:** Apakah ini tiket berbayar (revenue ticket) atau tiket penukaran poin (award/redemption ticket).

2.  **Ketidakpastian dalam Hubungan Ordinal dan Kuantitatif:**
    * Hubungan antara **`class`** dan **`harga`** menjadi tidak pasti. Secara teoretis dalam industri penerbangan, kelas tarif yang berbeda memiliki tingkat harga yang berbeda pula. Namun, tanpa definisi, kita tidak dapat memvalidasi apakah kelas 'Y' secara konsisten lebih mahal daripada kelas 'Q'. Analisis apa pun yang mencoba membangun hubungan antara kelas dan harga harus didasarkan pada asumsi yang diturunkan dari data itu sendiri (misalnya, dengan menghitung harga rata-rata per kelas), bukan dari aturan bisnis yang telah ditetapkan, sehingga mengurangi validitas kesimpulan.

3.  **Keterbatasan pada Kedalaman Analisis Strategis:**
    * Ketiadaan metadata menghalangi analisis strategis yang mendalam, seperti:
        * **Analisis *Yield Management*:** Tidak mungkin secara akurat menganalisis strategi harga maskapai atau *yield* per sektor jika kita tidak dapat membedakan antara tiket diskon dan tiket tarif penuh.
        * **Segmentasi Pelanggan:** Upaya untuk melakukan segmentasi penumpang berdasarkan kombinasi `class`, `agent_type`, dan `sector` akan bersifat dangkal karena makna sebenarnya dari segmen-segmen ini tidak diketahui.
        * **Analisis *Booking Window*:** Menghitung rentang waktu antara `generation_date` dan `travel_date` adalah mungkin, tetapi menghubungkannya dengan perilaku pemesanan menjadi sulit tanpa mengetahui jenis tarif (`class`) yang dibeli.

4.  **Risiko dalam Pra-pemrosesan dan Pemodelan:**
    * Keputusan untuk melakukan *encoding* pada variabel `class` menjadi sangat problematis. Memperlakukannya sebagai variabel nominal murni (melalui *one-hot encoding*) akan mengabaikan potensi hirarki harga, sementara memaksakan urutan ordinal akan sepenuhnya didasarkan pada spekulasi dan berisiko memasukkan bias yang signifikan ke dalam model prediktif.

Dengan demikian, temuan dari analisa ini harus dianggap sebagai eksplorasi awal terhadap pola-pola yang ada dalam data. Untuk analisis yang lebih definitif dan kesimpulan yang dapat ditindaklanjuti secara komersial, diperlukan validasi dan pengayaan data dengan kamus data resmi dari penyedia data.

## Preparation

In [1]:
import os
import dill
import py7zr
import pickle
import logging
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
from datetime import datetime
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

dirloc = os.getcwd()

In [2]:
# Load data
df = pd.read_csv(f'{dirloc}/bahan_test.csv')

print("=== COMPREHENSIVE DATA CLEANING REPORT ===")
print(f"Analysis Date: 2025-06-11 04:55:51 UTC")
print(f"Analyst: Mochammad Daffa Putra Karyudi")
print(f"Dataset shape: {df.shape}")
display(df.head())

=== COMPREHENSIVE DATA CLEANING REPORT ===
Analysis Date: 2025-06-11 04:55:51 UTC
Analyst: Mochammad Daffa Putra Karyudi
Dataset shape: (3648142, 13)


,order_type,pax_name,order_id,generation_date,travel_date,status,agent,region,agent_type,airline,harga,class,sector
0,Domestic,f1473af8,PNR1,20181009,20181011,To Deliver,5770,DKI Jakarta,agent,XG,0.001324,X,CGK-PDG
1,Domestic,58475d2a,PNR2,20181009,20181010,To Deliver,5770,DKI Jakarta,agent,XG,0.001856,X,PKU-CGK
2,Domestic,000f89da,PNR3,20181009,20181027,To Deliver,5770,DKI Jakarta,agent,LE,0.002189,Q,LOP-CGK
3,Domestic,b78d5043,PNR4,20181009,20181010,To Deliver,5770,DKI Jakarta,agent,MI,0.001609,V,CGK-KNO
4,Domestic,bbaa07cd,PNR5,20181009,20190103,To Deliver,5770,DKI Jakarta,agent,AZ,0.001340,L,CGK-PGK


## Data Wrangling

In [3]:
# 1. Missing Values Check
print("\n1. MISSING VALUES ANALYSIS:")
missing_summary = df.isnull().sum()
missing_percentage = (missing_summary / len(df)) * 100
missing_report = pd.DataFrame({
    'Column': missing_summary.index,
    'Missing_Count': missing_summary.values,
    'Missing_Percentage': missing_percentage.values
})
display(missing_report)


1. MISSING VALUES ANALYSIS:


,Column,Missing_Count,Missing_Percentage
0,order_type,0,0.000000
1,pax_name,0,0.000000
2,order_id,0,0.000000
3,generation_date,0,0.000000
4,travel_date,0,0.000000
5,status,0,0.000000
6,agent,0,0.000000
7,region,205627,5.636486
8,agent_type,0,0.000000
9,airline,0,0.000000


In [4]:
# 2. Duplicate Analysis
print("\n2. DUPLICATE ANALYSIS:")
total_duplicates = df.duplicated().sum()
print(f"Total duplicate rows: {total_duplicates}")


2. DUPLICATE ANALYSIS:
Total duplicate rows: 39


### Data Cleaning

In [5]:
with open('clean_data.pkl', 'rb') as f:
    clean_function = dill.load(f)

# Define file paths
csv_path = os.path.join(dirloc, 'bahan_cleaned.csv')
archive_path = os.path.join(dirloc, 'bahan_cleaned.7z')

# 2. Check if the cleaned CSV already exists. If not, create it.
if not os.path.exists(csv_path):
    print(f"'{csv_path}' not found. Cleaning data and creating the CSV file...")
    
    # Run the cleaning function
    df_cleaned = clean_function(df, dirloc)
    
    # Save the cleaned dataframe to CSV
    df_cleaned.to_csv(csv_path, index=False)
    print(f"Successfully created '{csv_path}'.")
else:
    df_cleaned = pd.read_csv(csv_path)
    print(f"Skipping CSV creation: '{csv_path}' already exists.")


# 3. Check if the 7z archive already exists. If not, create it.
if not os.path.exists(archive_path):
    # Also check if the source CSV file exists before trying to archive it
    if os.path.exists(csv_path):
        print(f"'{archive_path}' not found. Archiving the CSV file...")
        
        # Create the 7z archive
        with py7zr.SevenZipFile(archive_path, 'w') as archive:
            archive.write(csv_path, os.path.basename(csv_path))
            
        print(f"Successfully created archive '{archive_path}'.")
    else:
        # This case would be unusual if the logic runs sequentially, but it's good practice
        print(f"Could not create archive because source file '{csv_path}' does not exist.")
else:
    print(f"Skipping archiving: '{archive_path}' already exists.")

2025-06-12 12:45:24,774 - INFO - INITIATING COMPREHENSIVE AIRLINE DATA CLEANING PIPELINE
2025-06-12 12:45:24,775 - INFO - Execution timestamp: 2025-06-12 05:45:24 UTC
2025-06-12 12:45:24,775 - INFO - Executed by: Mochammad Daffa Putra Karyudi
2025-06-12 12:45:24,776 - INFO - Initial dataset dimensions: 3,648,142 rows × 13 columns


'/home/keycode/Coding/github/Case-Study-Assignment/bahan_cleaned.csv' not found. Cleaning data and creating the CSV file...


2025-06-12 12:45:24,992 - INFO - Phase 1: Standardizing invalid region values to NULL
2025-06-12 12:45:26,007 - INFO - Invalid region values identified:
2025-06-12 12:45:26,007 - INFO -   Value '-1': 53,038 occurrences
2025-06-12 12:45:26,007 - INFO -   Value 'Indonesia': 84 occurrences
2025-06-12 12:45:26,008 - INFO -   Value 'indonesia': 24 occurrences
2025-06-12 12:45:26,008 - INFO -   Value 'INDONESIA': 24 occurrences
2025-06-12 12:45:26,251 - INFO - Phase 1 Complete - 53,170 records standardized to NULL
2025-06-12 12:45:26,251 - INFO - Total NULL regions: 258,797
2025-06-12 12:45:26,252 - INFO - Phase 2: Implementing region data formatting standards
2025-06-12 12:45:26,488 - INFO - Unique non-null regions before standardization: 96
2025-06-12 12:45:30,579 - INFO - Basic string formatting applied successfully
2025-06-12 12:45:30,580 - INFO - Phase 3: Applying Indonesian region standardization mapping
2025-06-12 12:45:30,580 - INFO - Region mapping successfully loaded from: /home/ke

Successfully created '/home/keycode/Coding/github/Case-Study-Assignment/bahan_cleaned.csv'.
'/home/keycode/Coding/github/Case-Study-Assignment/bahan_cleaned.7z' not found. Archiving the CSV file...
Successfully created archive '/home/keycode/Coding/github/Case-Study-Assignment/bahan_cleaned.7z'.


#### Dokumentasi Teknis: Pipeline Pembersihan Data Airline `clean_airline_dataset_comprehensive`

---

##### Fase 1: Standarisasi Awal Nilai Region
* **Tujuan:** Mengidentifikasi dan menetralkan nilai pada kolom `region` yang secara definitif tidak valid atau merupakan placeholder.
* **Asumsi yang Mendasari:** Nilai-nilai seperti `-1`, `Indonesia`, `nan`, dan `NULL` tidak mengandung informasi geografis yang spesifik dan dapat dianalisis. Nilai-nilai ini dianggap sebagai hasil dari kesalahan input atau placeholder sistem, yang secara fungsional setara dengan data yang hilang (*missing value*).
* **Langkah Eksekusi:**
    1.  Skrip mengidentifikasi `region` dengan nilai-nilai dari daftar `invalid_region_identifiers`.
    2.  Jumlah kemunculan setiap nilai tidak valid didokumentasikan melalui logging.
    3.  Semua nilai yang teridentifikasi diubah menjadi `np.nan` (NULL) untuk diproses lebih lanjut pada fase imputasi.

##### Fase 2: Pemformatan String Region
* **Tujuan:** Menyeragamkan format penulisan untuk semua data `region` yang tersisa guna memastikan konsistensi kategorikal.
* **Asumsi yang Mendasari:** Konsistensi format penulisan (tanpa spasi berlebih dan menggunakan "Title Case") adalah prasyarat untuk pemetaan dan pengelompokan yang akurat. Tanpa ini, nilai seperti `'jawa barat'` dan `'Jawa Barat'` akan dianggap sebagai dua kategori yang berbeda.
* **Langkah Eksekusi:**
    1.  Menghapus spasi di awal dan akhir string (`.str.strip()`).
    2.  Mengganti spasi ganda atau lebih di tengah string dengan satu spasi (`.str.replace(r'\s+', ' ', regex=True)`).
    3.  Mengonversi seluruh string ke format "Title Case" (`.str.title()`).

##### Fase 3: Implementasi Pemetaan Region Indonesia
* **Tujuan:** Menerapkan aturan pemetaan yang kompleks untuk mengonsolidasikan berbagai variasi nama region ke standar nama provinsi.
* **Asumsi yang Mendasari:** File `indonesian_region_mapping_v1.1.pkl` adalah "sumber kebenaran" (*single source of truth*). Kamus di dalamnya, yang dibuat dari analisis sebelumnya, secara akurat memetakan variasi nama (termasuk singkatan, kesalahan ketik, dan nama level kota) ke nama provinsi yang standar.
* **Langkah Eksekusi:**
    1.  Pipeline memuat kamus pemetaan dari file `.pkl` yang telah disiapkan.
    2.  Fungsi `standardize_region` diaplikasikan ke setiap baris pada kolom `region`.
    3.  Fungsi ini mencocokkan nilai region dengan kunci dalam kamus. Jika ditemukan, nilainya akan diperbarui. Jika tidak, nilai asli dipertahankan.

##### Fase 4: Strategi Imputasi Region yang Hilang
* **Tujuan:** Mengisi nilai `region` yang kosong (`np.nan`) secara cerdas menggunakan informasi dari data itu sendiri.
* **Asumsi yang Mendasari:** Perilaku bisnis seorang agen tiket (`agent`) bersifat konsisten secara geografis. Oleh karena itu, modus (region yang paling sering muncul) dari data historis seorang agen adalah prediktor yang paling andal untuk mengisi data region yang hilang pada transaksi lain dari agen yang sama.
* **Langkah Eksekusi:**
    1.  Skrip membuat pemetaan dinamis dengan mengelompokkan data berdasarkan `agent` dan mencari modus `region` untuk setiap agen.
    2.  Fungsi `apply_region_imputation` diterapkan. Jika sebuah baris memiliki `region` kosong, fungsi ini mengisinya dengan modus region dari `agent` yang bersangkutan.
    3.  Jika modus tidak ditemukan, region diisi dengan nilai `'Unknown Region'` untuk menandakan bahwa imputasi tidak memungkinkan.

##### Fase 5: Pemrosesan Komprehensif `class` Penerbangan
* **Tujuan:** Membersihkan kolom `class` yang sangat terpolusi melalui kombinasi penghapusan data yang tidak dapat diperbaiki, pemetaan variasi, dan imputasi cerdas.
* **Asumsi yang Mendasari:**
    * **Untuk Penghapusan:** Entri seperti `Promo`, `DEPOSIT`, `-`, dan kode acak lainnya bukan merupakan *fare class* yang valid. Nilai-nilai ini adalah *noise* yang tidak dapat diselamatkan, dan menghapus seluruh baris data adalah pendekatan yang lebih baik daripada melakukan imputasi yang sangat spekulatif.
    * **Untuk Pemetaan:** Variasi seperti `v`, `A/0`, dan `E1` adalah representasi non-standar namun valid dari kode IATA induknya dan dapat dikonsolidasikan tanpa kehilangan makna.
    * **Untuk Imputasi:** Terdapat korelasi kuat antara `harga` tiket dan `class`-nya. Oleh karena itu, harga adalah fitur terbaik yang tersedia untuk mengestimasi `class` yang hilang, terutama jika dikombinasikan dengan `airline`. Kelas `'Y'` adalah *fallback* yang paling umum dan aman.
* **Langkah Eksekusi:**
    1.  **Fase 5a (Penghapusan Ketat):** Memuat daftar `entries_to_delete` dan menghapus seluruh baris data yang `class`-nya cocok dengan salah satu entri dalam daftar tersebut.
    2.  **Fase 5b (Pemetaan Standarisasi):** Menerapkan kamus `flight_class_mapping` untuk menstandarisasi variasi `class` yang valid ke kode IATA induknya.
    3.  **Fase 5c (Imputasi Berbasis Harga):** Untuk `class` yang masih kosong, pipeline mengisinya dengan mencari kelas pada maskapai yang sama yang memiliki median harga paling mendekati.

##### Fase 6: Eliminasi Data Duplikat
* **Tujuan:** Menghapus baris data yang identik secara absolut untuk memastikan setiap baris unik.
* **Asumsi yang Mendasari:** Baris data yang 100% identik adalah hasil dari kesalahan teknis dalam proses pengumpulan data (misalnya, duplikasi saat proses ETL) dan tidak mengandung informasi baru yang bernilai.
* **Langkah Eksekusi:**
    1.  Fungsi `df.drop_duplicates()` dipanggil pada keseluruhan dataset.
    2.  Jumlah baris duplikat yang dihapus dicatat untuk dokumentasi.

##### Fase 7: Validasi Integritas Data
* **Tujuan:** Melakukan pemeriksaan kewajaran (*sanity check*) akhir untuk memastikan dataset mematuhi aturan bisnis yang fundamental.
* **Asumsi yang Mendasari:** Transaksi penerbangan yang valid secara logis harus memiliki harga positif dan tanggal perjalanan yang tidak mendahului tanggal pemesanan. Nilai di luar batas kewajaran (seperti `booking_lead_time > 365 hari`) dianggap sebagai anomali.
* **Langkah Eksekusi:**
    1.  Kolom sementara `booking_lead_time` dibuat.
    2.  Dataset diperiksa untuk kondisi-kondisi yang melanggar aturan bisnis (misal: `harga < 0`).
    3.  Jumlah pelanggaran dicatat dalam log sebagai indikator kesehatan data akhir.
    4.  Kolom sementara dihapus setelah validasi selesai.

### Data Validation

In [6]:
# 4. Business Logic Validation
print("\n4. BUSINESS LOGIC VALIDATION:")
df_clean = df_cleaned.copy()

# Convert dates with error handling
df_clean['generation_date'] = pd.to_datetime(df_clean['generation_date'], format='%Y%m%d', errors='coerce')
df_clean['travel_date'] = pd.to_datetime(df_clean['travel_date'], format='%Y%m%d', errors='coerce')

# Check date conversion success
date_conversion_issues = {
    'generation_date_invalid': df_clean['generation_date'].isnull().sum(),
    'travel_date_invalid': df_clean['travel_date'].isnull().sum()
}
print(f"Date conversion issues: {date_conversion_issues}")

df_clean['lead_time'] = (df_clean['travel_date'] - df_clean['generation_date']).dt.days

# Check for logical inconsistencies
logic_issues = {
    'negative_lead_time': (df_clean['lead_time'] < 0).sum(),
    'negative_prices': (df_clean['harga'] < 0).sum(),
    'extreme_lead_time': (df_clean['lead_time'] > 365).sum(),
    'zero_prices': (df_clean['harga'] == 0).sum()
}
print(f"Logic issues found: {logic_issues}")


4. BUSINESS LOGIC VALIDATION:
Date conversion issues: {'generation_date_invalid': 0, 'travel_date_invalid': 0}
Logic issues found: {'negative_lead_time': 1191, 'negative_prices': 0, 'extreme_lead_time': 9, 'zero_prices': 2}


In [7]:
# 5. Text Data Standardization
print("\n5. TEXT DATA STANDARDIZATION:")

# Check for potential inconsistencies in categorical data
categorical_analysis = {}
for col in ['order_type', 'status', 'agent_type', 'airline', 'class', 'region']:
    unique_values = df_clean[col].unique()
    categorical_analysis[col] = {
        'unique_count': len(unique_values),
        'values': list(unique_values)
    }
    print(f"{col}: {len(unique_values)} unique values")


5. TEXT DATA STANDARDIZATION:
order_type: 2 unique values
status: 3 unique values
agent_type: 5 unique values
airline: 162 unique values
class: 26 unique values
region: 36 unique values


In [8]:
# 6. Outlier Detection
print("\n6. OUTLIER DETECTION:")
price_stats = df_clean['harga'].describe()
Q1 = df_clean['harga'].quantile(0.25)
Q3 = df_clean['harga'].quantile(0.75)
IQR = Q3 - Q1
outlier_bounds = {
    'lower_bound': Q1 - 1.5 * IQR,
    'upper_bound': Q3 + 1.5 * IQR
}
outliers = ((df_clean['harga'] < outlier_bounds['lower_bound']) | 
            (df_clean['harga'] > outlier_bounds['upper_bound'])).sum()
print(f"Price outliers detected: {outliers}")
print(f"Outlier bounds: {outlier_bounds}")


6. OUTLIER DETECTION:
Price outliers detected: 230854
Outlier bounds: {'lower_bound': 0.0001048852543764499, 'upper_bound': 0.00305932954337925}


In [9]:
# 7. Data Quality Score
print("\n7. DATA QUALITY SCORE:")
total_records = len(df_clean)
missing_summary = df_clean.isnull().sum()
total_duplicates = df_clean.duplicated().sum()
logic_issues = {
    'negative_lead_time': (df_clean['lead_time'] < 0).sum(),
    'negative_prices': (df_clean['harga'] < 0).sum(),
    'extreme_lead_time': (df_clean['lead_time'] > 365).sum(),
    'zero_prices': (df_clean['harga'] == 0).sum()
}
quality_metrics = {
    'completeness': (1 - missing_summary.sum() / (total_records * len(df_clean.columns))) * 100,
    'uniqueness': (1 - total_duplicates / total_records) * 100,
    'validity': (1 - sum(logic_issues.values()) / total_records) * 100,
    'consistency': 95  # Based on categorical data review
}
overall_quality = np.mean(list(quality_metrics.values()))
print(f"Data Quality Metrics: {quality_metrics}")
print(f"Overall Data Quality Score: {overall_quality:.2f}%")

print(f"\nCleaned dataset ready for analysis: {df_clean.shape}")


7. DATA QUALITY SCORE:
Data Quality Metrics: {'completeness': 100.0, 'uniqueness': 100.0, 'validity': 99.96692280851222, 'consistency': 95}
Overall Data Quality Score: 98.74%

Cleaned dataset ready for analysis: (3633924, 14)


## Exploratory Data Analysis

### Q1 What are top 10 distribution of orders by airline?

In [10]:
# Question 1: Distribution of orders by airline (Top 10)
print("Q1: What are top 10 distribution of orders by airline?")
airline_dist = df_cleaned['airline'].value_counts().head(10).reset_index()
display(airline_dist)
airline_dist.columns = ['airline', 'order_count']
fig1 = px.pie(airline_dist, values='order_count', names='airline', 
              title="Q1: Order Distribution by Airline (Top 10)")
fig1.show()

Q1: What are top 10 distribution of orders by airline?


,index,airline
0,XG,1211577
1,AZ,486441
2,LE,485577
3,MI,353668
4,GF,274180
5,HV,170205
6,DB,146965
7,QT,77259
8,TS,68185
9,TI,59343


#### Wawasan dan Interpretasi (Insights)

1.  **Dominasi Pasar yang Jelas oleh Satu Maskapai (Market Dominance)**
    * Maskapai **XG** mendominasi pasar secara signifikan dengan pangsa **36.3%**. Ini menunjukkan bahwa lebih dari sepertiga dari total pesanan di antara 10 maskapai teratas dikuasai oleh satu pemain. Hal ini bisa disebabkan oleh berbagai faktor seperti jaringan rute yang luas, harga yang kompetitif, kapasitas penerbangan yang besar, atau loyalitas pelanggan yang tinggi.

2.  **Persaingan Ketat di Peringkat Kedua (Tier 2 Competition)**
    * Maskapai **AZ** dan **LE** bersaing sangat ketat di posisi kedua, masing-masing memegang pangsa pasar **14.6%**. Jika digabungkan, kekuatan mereka (29.2%) hampir menyaingi pemimpin pasar. Ini menandakan adanya persaingan yang sehat dan tidak ada monopoli tunggal di level ini. Strategi mereka kemungkinan besar sangat mirip, baik dari segi harga maupun target pasar.

3.  **Pemain Menengah yang Solid (Mid-Tier Players)**
    * Maskapai **MI (10.6%)** dan **GF (8.23%)** merupakan pemain menengah yang solid. Mereka memiliki pangsa pasar yang cukup untuk tetap relevan dan kemungkinan besar memiliki ceruk pasar (niche market) atau keunggulan di rute-rute tertentu.

4.  **Pemain Ceruk dan Pelengkap (Niche & Minor Players)**
    * Maskapai **HV, TI, DB, QT, dan TS** secara kolektif hanya menyumbang sekitar **15.7%** dari total pesanan. Mereka adalah pemain yang lebih kecil dalam konteks 10 besar ini. Kemungkinan mereka fokus pada:
        * **Rute spesifik** yang tidak banyak dilayani oleh maskapai besar.
        * **Model bisnis tertentu** (misalnya, LCC - Low-Cost Carrier atau maskapai regional).
        * **Target demografis** yang lebih sempit.

---

#### Kesimpulan Analisis

Distribusi pesanan ini menunjukkan struktur pasar yang **terkonsentrasi di puncak (Top-Heavy Market)**. Satu maskapai (XG) adalah pemimpin yang tak terbantahkan, diikuti oleh dua pesaing kuat (AZ & LE), dan sisanya adalah pemain menengah atau ceruk.

### Q2 What are Top 10 average ticket prices by airline


In [11]:
# Question 2: Average ticket prices by airline (Top 10)
print("Q2: What are Top 10 average ticket prices by airline")
avg_price_airline = df_cleaned.groupby('airline')['harga'].mean().reset_index()

# Sort by average price from highest to lowest and get top 10
avg_price_airline = avg_price_airline.sort_values('harga', ascending=False).head(10)

# Create horizontal bar chart
fig2 = px.bar(avg_price_airline, x='harga', y='airline',
              title="Q2: Average Ticket Prices by Airline (Top 10)",
              orientation='h')  # 'h' for horizontal

# Display the table as well
display(avg_price_airline.style.hide(axis="index"))

# Update layout to improve readability
fig2.update_layout(
    yaxis={'categoryorder': 'total ascending'},  # This ensures the sorting is maintained
    xaxis_title="Average Price (Harga)",
    yaxis_title="Airline",
    width=1000,  # Increase width
    height=600   # Reduced height since we only have 10 bars now
)

fig2.show()

Q2: What are Top 10 average ticket prices by airline


airline,harga
KD,0.011104
BO,0.010003
WZ,0.008826
PC,0.007963
DS,0.007639
CY,0.007532
ZS,0.007331
GJ,0.006914
JG,0.006868
LC,0.006651


#### Observasi dan Interpretasi

**1. Observasi Teknis Penting: Harga Ternormalisasi**
Sebelum masuk ke interpretasi bisnis, penting untuk dicatat bahwa nilai pada kolom `harga` tampaknya telah **dinormalisasi atau diskalakan** (menghasilkan nilai desimal yang sangat kecil). Ini adalah praktik umum dalam pengolahan data. Artinya, kita sedang membandingkan **nilai relatif** antar maskapai, bukan harga absolut dalam Rupiah. Maskapai `KD` memiliki harga rata-rata hampir dua kali lipat lebih tinggi dari `LC` dalam skala ini.

**2. Wawasan Paling Krusial: Perbandingan dengan Volume Pesanan (Q1)**
Inilah temuan yang paling signifikan: **Tidak ada satu pun maskapai dari daftar 10 teratas berdasarkan volume pesanan (Q1) yang muncul di daftar 10 teratas berdasarkan harga rata-rata tertinggi ini.**

* **Maskapai Populer (Q1):** XG, AZ, LE, MI, dll. (volume tinggi).
* **Maskapai Mahal (Q2):** KD, BO, WZ, PC, dll. (harga tinggi).

Ini dengan jelas menunjukkan adanya **dua strategi bisnis yang berbeda** di pasar:
* **Strategi Volume Tinggi, Harga Rendah:** Maskapai seperti **XG, AZ, dan LE** kemungkinan besar adalah *Low-Cost Carrier (LCC)* atau pemain besar yang fokus pada kelas ekonomi untuk menarik jumlah penumpang sebanyak mungkin.
* **Strategi Volume Rendah, Harga Tinggi:** Maskapai dalam daftar ini (**KD, BO, WZ**, dst.) kemungkinan adalah *Full-Service Carrier*, maskapai premium, atau maskapai yang melayani rute-rute khusus/bisnis di mana mereka bisa menetapkan harga yang lebih tinggi.

---

#### Hipotesis

Berdasarkan temuan ini, merumuskan beberapa hipotesis yang perlu divalidasi:

1.  **Hipotesis:** Maskapai seperti `KD` dan `BO` memiliki proporsi penjualan tiket kelas **bisnis/utama** yang lebih tinggi.
2.  **Hipotesis:** Maskapai seperti `XG` dan `AZ` (dari Q1) hampir secara eksklusif menjual tiket kelas **ekonomi**.

---

#### Kesimpulan
Analisis ini mengungkap adanya segmen pasar premium yang dilayani oleh sekelompok maskapai tertentu. Pemosisian harga mereka secara signifikan lebih tinggi daripada rata-rata pasar, menandakan strategi bisnis yang berfokus pada nilai dan kualitas, bukan pada volume.

### Q3 Which regions generate the most bookings?

In [12]:
# Question 3: Regional booking analysis
print("Q3: Which regions generate the most bookings?")
region_bookings = df_cleaned['region'].value_counts().reset_index()
region_bookings.columns = ['region', 'bookings']

# Sort by bookings from highest to lowest
region_bookings = region_bookings.sort_values('bookings', ascending=False)

display(region_bookings.style.hide(axis="index"))

fig3 = px.bar(region_bookings, x='region', y='bookings',
              title="Q3: Bookings by Region")

# Correct way to update x-axis properties
fig3.update_layout(xaxis_tickangle=45)
fig3.show()

Q3: Which regions generate the most bookings?


region,bookings
DKI Jakarta,1722103
Jawa Barat,282461
Sumatera Utara,192034
Kalimantan Timur,144070
Jawa Timur,107963
Riau,102311
Unknown Region,96008
Banten,95138
Bali,94713
Sumatera Selatan,88728


#### Temuan Utama

1. Temuan Utama (Key Findings):
Dominasi Absolut Satu Wilayah: DKI Jakarta adalah pusat pasar yang dominan secara absolut, dengan total 1.72 juta pemesanan. Angka ini menciptakan jarak yang sangat besar dengan wilayah peringkat kedua, menunjukkan peran Jakarta sebagai hub utama yang tak tertandingi.

2. Pasar Sekunder yang Signifikan: Meskipun ada jarak yang jauh, Jawa Barat (282 ribu), Sumatera Utara (192 ribu), Kalimantan Timur (144 ribu), dan Jawa Timur (107 ribu) merupakan pasar sekunder yang sangat penting dan merepresentasikan kantong-kantong permintaan yang kuat.

3. Pentingnya Validitas Data: Terdapat kategori "Unknown Region" dengan volume pemesanan yang cukup tinggi (96 ribu). Hal ini menunjukkan adanya potensi kehilangan informasi yang dapat memengaruhi akurasi pemetaan geografis secara keseluruhan. Perbaikan kualitas pencatatan data diperlukan.

4. Distribusi Ekor Panjang (Long Tail): Setelah 5-10 wilayah teratas, volume pemesanan per wilayah menurun secara drastis, menunjukkan adanya banyak wilayah dengan kontribusi yang lebih kecil (ekor panjang).
___

#### Kesimpulan
Distribusi pasar secara geografis sangat terkonsentrasi di DKI Jakarta. Namun, terdapat beberapa pasar sekunder yang kuat yang menjadi pilar penting bagi bisnis. Peningkatan kualitas data regional akan semakin mempertajam pemahaman tentang lanskap pasar ini.

### Q4: What are the booking patterns over travel dates?

In [ ]:
# Question 4: Booking patterns over travel dates
print("Q4: What are the booking patterns over travel dates?")
travel_date_pattern = df_cleaned.groupby('travel_date').size().reset_index(name='bookings')
# Display all rows in the DataFrame
with pd.option_context('display.max_rows', None):
    print(travel_date_pattern.to_string(index=False))
fig4 = px.line(travel_date_pattern, x='travel_date', y='bookings',
               title="Q4: Booking Patterns Over Travel Dates")
fig4.show()

Q4: What are the booking patterns over travel dates?
travel_date  bookings
 2017-01-01         5
 2017-01-03         3
 2017-01-04         2
 2017-01-07         1
 2017-01-09         1
 2017-01-24         4
 2017-01-29         4
 2017-02-02         3
 2017-02-05         3
 2017-02-09         3
 2017-02-11         1
 2017-02-12         1
 2017-02-15         1
 2017-02-16         2
 2017-02-19         1
 2017-04-01       255
 2017-04-02       919
 2017-04-03      1408
 2017-04-04      2224
 2017-04-05      2638
 2017-04-06      2787
 2017-04-07      3006
 2017-04-08      2951
 2017-04-09      3177
 2017-04-10      3662
 2017-04-11      3491
 2017-04-12      3869
 2017-04-13      3927
 2017-04-14      3556
 2017-04-15      3552
 2017-04-16      3597
 2017-04-17      4240
 2017-04-18      4002
 2017-04-19      4481
 2017-04-20      4561
 2017-04-21      4415
 2017-04-22      3992
 2017-04-23      4221
 2017-04-24      3799
 2017-04-25      4733
 2017-04-26      4986
 2017-04-27      4564
 

#### **Temuan Utama**

Analisis data time-series menunjukkan empat fase yang berbeda dalam siklus hidup pemesanan selama periode yang diamati:

1. **Fase Permulaan (Januari 2017 - Maret 2017):**
   * Aktivitas pemesanan sangat rendah dan sporadis. Periode ini kemungkinan menandai awal dari pengumpulan data atau fase awal peluncuran bisnis di mana volume masih sangat kecil.

2. **Fase Pertumbuhan dan Puncak (April 2017 - Desember 2018):**
   * Dimulai pada April 2017, terjadi lonjakan volume pemesanan yang signifikan, menandai dimulainya aktivitas bisnis yang sesungguhnya.
   * Terdapat **tren pertumbuhan yang jelas** dari pertengahan 2017 hingga akhir 2018.
   * **Pola musiman (seasonality)** sangat terlihat jelas, dengan puncak-puncak permintaan yang dapat diidentifikasi:
     * **Puncak Pertengahan Tahun (Juni 2018):** Lonjakan permintaan yang sangat kuat terjadi di sekitar bulan Juni 2018. Ini sangat mungkin berkorelasi dengan periode libur **Lebaran (Idul Fitri)**, yang merupakan musim puncak perjalanan domestik di Indonesia.
     * **Puncak Akhir Tahun (Desember 2018):** Titik tertinggi absolut dari seluruh periode data terjadi pada akhir Desember 2018. Puncak ini secara jelas berhubungan dengan musim liburan **Natal dan Tahun Baru**, dengan tanggal perjalanan tertinggi pada **21 Desember 2018 (13.862 pemesanan)**.

3. **Fase Penurunan Drastis (Januari 2019 - Maret 2019):**
   * Segera setelah mencapai puncaknya di awal Januari 2019, volume pemesanan mengalami **penurunan yang sangat tajam dan drastis**. Ini bukan sekadar penurunan musiman biasa, melainkan sebuah "crash" yang menandakan adanya perubahan struktural.
   * Volume jatuh dari ribuan pemesanan per hari menjadi hanya ratusan dalam waktu singkat.

4. **Fase Aktivitas Rendah (April 2019 dan seterusnya):**
   * Setelah penurunan drastis, volume pemesanan stabil pada tingkat yang sangat rendah, jauh di bawah level sebelum puncak. Ini menunjukkan "kondisi normal baru" bagi bisnis atau data yang tercatat.

---
#### **Kesimpulan dan Implikasi Strategis**

* Bisnis ini menunjukkan pola musiman yang dapat diprediksi selama periode puncaknya (2017-2018), terutama terkait dengan libur Lebaran dan Akhir Tahun. Ini adalah informasi krusial untuk perencanaan kapasitas, stok, dan strategi penetapan harga.
* **Peristiwa paling kritikal** dalam data ini adalah **penurunan drastis pada awal tahun 2019**. Ini adalah anomali yang paling signifikan dan memerlukan investigasi mendalam. Kemungkinan penyebabnya bisa meliputi:
  * **Faktor Eksternal:** Krisis ekonomi, bencana alam, atau peristiwa besar lain yang menekan permintaan perjalanan.
  * **Perubahan Internal:** Perubahan strategi bisnis, penghentian rute/layanan populer, atau perubahan model harga.
  * **Isu Teknis:** Masalah pada sistem pengumpulan data yang menyebabkan data tidak tercatat dengan benar setelah Januari 2019.

Tanpa memahami penyebab penurunan ini, setiap upaya peramalan (forecasting) di masa depan akan sangat tidak akurat.
    

### Q5: What is the booking trend over generation dates?

In [14]:
# Question 5: Generation date booking trends
print("Q5: What is the booking trend over generation dates?")
generation_trend = df_cleaned.groupby('generation_date').size().reset_index(name='bookings')
with pd.option_context('display.max_rows', None):
    print(generation_trend.to_string(index=False))
fig5 = px.line(generation_trend, x='generation_date', y='bookings',
                title="Q5: Daily Booking Generation Trend")
fig5.show()

Q5: What is the booking trend over generation dates?
generation_date  bookings
     2017-03-04         2
     2017-03-05         6
     2017-03-11         4
     2017-03-14         2
     2017-03-15        25
     2017-03-16         2
     2017-03-17         8
     2017-03-20        15
     2017-03-21         7
     2017-03-22         6
     2017-03-23        62
     2017-03-24        40
     2017-03-25        18
     2017-03-26        12
     2017-03-27        57
     2017-03-28        75
     2017-03-29       237
     2017-03-30       261
     2017-03-31       564
     2017-04-01      4084
     2017-04-02      2748
     2017-04-03      5589
     2017-04-04      5694
     2017-04-05      5645
     2017-04-06      6560
     2017-04-07      6407
     2017-04-08      4740
     2017-04-09      3034
     2017-04-10      6632
     2017-04-11      6646
     2017-04-12      6191
     2017-04-13      6463
     2017-04-14      4113
     2017-04-15      4184
     2017-04-16      2757
     2017-0

#### **Temuan Utama**

1.  **Pola Mingguan yang Sangat Kuat (Strong Weekly Cyclicality):**
    * Grafik menunjukkan pola naik-turun yang sangat teratur dan berulang setiap minggu. Volume pemesanan secara konsisten **tinggi pada hari kerja (Senin-Jumat)** dan **turun secara signifikan pada akhir pekan (Sabtu-Minggu)**.
    * Pola ini sangat umum untuk bisnis yang melayani segmen korporat atau B2B, di mana keputusan pembelian sebagian besar terjadi selama jam kerja. Ini juga bisa mengindikasikan bahwa kampanye pemasaran atau aktivitas tim penjualan paling aktif pada hari kerja.

2.  **Periode Penjualan Puncak (Peak Sales Periods):**
    * Terdapat beberapa periode di mana volume pemesanan harian melonjak secara masif, jauh di atas rata-rata normal. Puncak tertinggi terjadi pada **akhir 2018**, khususnya di **Desember**.
    * **Puncak Tertinggi (5 Desember 2018):** Pada tanggal ini, tercatat **21.941 pemesanan**, yang merupakan aktivitas penjualan harian tertinggi dalam seluruh dataset. Puncak-puncak lain yang signifikan juga terjadi di sekitar periode ini, seperti **7 Desember 2018 (21.298 pemesanan)**.
    * Lonjakan luar biasa ini kemungkinan besar bukan disebabkan oleh permintaan organik, melainkan oleh **acara penjualan besar (major sales events)**, seperti promo akhir tahun, travel fair online, atau kampanye promosi besar-besaran.

3.  **Perbedaan Kunci dengan Tren Tanggal Perjalanan (Q4):**
    * **Booking Window:** Perbedaan antara grafik `generation_date` (kapan pesan) dan `travel_date` (kapan terbang) menunjukkan adanya "booking window" atau jeda waktu antara pemesanan dan perjalanan. Puncak pemesanan di awal Desember 2018 adalah untuk perjalanan di akhir Desember 2018 dan awal Januari 2019.
    * **Sifat Puncak:** Puncak pada `generation_date` lebih "tajam" dan sering kali didorong oleh promo (misalnya, promo gaji bulanan di tanggal 25-28 atau promo 10.10, 11.11, 12.12), sedangkan puncak pada `travel_date` lebih "lebar" dan mengikuti kalender liburan publik.
---
#### **Kesimpulan dan Implikasi Strategis**

* **Ritme Bisnis Didominasi Hari Kerja:** Strategi penjualan, alokasi sumber daya tim, dan jadwal kampanye email/digital marketing harus dioptimalkan untuk memaksimalkan momentum selama hari Senin hingga Jumat.
* **Keberhasilan Kampanye Penjualan Terukur:** Puncak-puncak masif pada `generation_date` membuktikan bahwa kampanye penjualan yang terfokus dan berbatas waktu sangat efektif dalam mendongkrak volume. Analisis lebih lanjut pada tanggal-tanggal puncak ini dapat memberikan wawasan tentang jenis promosi apa yang paling berhasil.
* **Manajemen Operasional:** Memahami pola mingguan ini krusial untuk manajemen staf. Tim layanan pelanggan dan operasional harus disiapkan untuk menangani volume yang lebih tinggi selama hari kerja.

Analisis `generation_date` ini memberikan pandangan langsung ke "inti" aktivitas penjualan harian perusahaan, melengkapi analisis musiman jangka panjang yang kita lihat dari `travel_date`.
    

### Q6: Which flight classes are most popular?

In [15]:
# Question 6: Flight class popularity
print("Q6: Which flight classes are most popular?")
class_popularity = df_cleaned['class'].value_counts().reset_index()
display(class_popularity.style.hide(axis="index"))
class_popularity.columns = ['flight_class', 'count']
fig6 = px.bar(class_popularity, x='flight_class', y='count',
              title="Q6: Flight Class Popularity")
fig6.show()

Q6: Which flight classes are most popular?


index,class
V,370266
X,323465
Q,314891
T,267718
N,255883
L,244257
M,228360
Y,206642
K,183550
H,173294


#### **Temuan Utama**

1.  **Memahami Kode Kelas (Fare Class):** Penting untuk dipahami bahwa kode satu huruf ini (V, X, Q, dll.) bukanlah kelas kabin utama (Ekonomi, Bisnis, First), melainkan **Kelas Tarif** atau **Booking Code**. Setiap kelas tarif memiliki harga dan aturan yang berbeda (misal: tingkat fleksibilitas, jatah bagasi, perolehan miles), meskipun berada dalam kabin yang sama.

2.  **Dominasi Kelas Tarif Ekonomi Promo:**
    * Kelas tarif **V (370k)**, **X (323k)**, **Q (314k)**, **T (267k)**, dan **N (255k)** adalah yang paling populer dengan selisih yang signifikan.
    * Dalam industri penerbangan, kelas-kelas tarif ini secara universal merepresentasikan berbagai tingkatan **tiket Kelas Ekonomi dengan harga diskon atau promo**. Volume penjualan yang sangat tinggi pada kelas-kelas ini mengonfirmasi bahwa mayoritas besar pelanggan sangat sensitif terhadap harga.

3.  **Distribusi Ekor Panjang (Long-Tail Distribution):**
    * Setelah 10-15 kelas teratas, jumlah pemesanan untuk setiap kelas tarif menurun secara drastis.
    * Kelas-kelas dengan volume lebih rendah seperti **J, C, F, dan D** secara tradisional sering dikaitkan dengan **Kelas Bisnis atau First Class** yang harganya jauh lebih mahal dan fleksibel. Jumlahnya yang sedikit sangat sesuai dengan ekspektasi pasar.
---
#### **Kesimpulan dan Implikasi Strategis**

* **Model Bisnis Berbasis Volume Ekonomi:** Data ini adalah bukti terkuat bahwa mesin utama pendapatan (dari segi volume) berasal dari penjualan tiket Kelas Ekonomi, khususnya pada tingkatan harga yang paling rendah. Strategi penetapan harga dan manajemen inventaris untuk kelas-kelas ini adalah yang paling krusial.
* **Peluang Segmentasi:** Keragaman kelas tarif menunjukkan adanya upaya maskapai untuk melakukan segmentasi pasar. Meskipun volume kelas premium (seperti J atau C) kecil, yield atau keuntungan per tiketnya bisa jadi jauh lebih tinggi. Analisis lebih lanjut bisa menggabungkan data ini dengan harga untuk menghitung profitabilitas per kelas tarif.
* **Wawasan untuk Pemasaran:** Kampanye pemasaran yang menargetkan audiens luas harus fokus pada promosi yang berkaitan dengan kelas-kelas tarif terpopuler (V, X, Q). Sementara itu, pemasaran untuk kelas premium harus lebih tersegmentasi dan menonjolkan nilai lebih (fleksibilitas, layanan) daripada sekadar harga.
    

### Q7: What are the top 10 most popular flight sectors?

In [16]:
# Question 7: Top flight sectors
print("Q7: What are the top 10 most popular flight sectors?")
top_sectors = df_cleaned['sector'].value_counts().head(10).reset_index()
display(top_sectors.style.hide(axis="index"))
top_sectors.columns = ['sector', 'frequency']
fig7 = px.bar(top_sectors, x='sector', y='frequency',
              title="Q7: Top 10 Flight Sectors")
fig7.show()

Q7: What are the top 10 most popular flight sectors?


index,sector
CGK-SUB,82357
DPS-CGK,80126
KNO-CGK,79510
SUB-CGK,77992
CGK-DPS,77428
CGK-KNO,72475
PDG-CGK,50643
CGK-PDG,49036
JOG-CGK,46685
CGK-JOG,46043


#### **Temuan Utama**

1.  **Dominasi Mutlak Jakarta (CGK) sebagai Super Hub:**
    * Fakta paling menonjol adalah bahwa **10 dari 10 rute teratas** melibatkan Bandara Internasional Soekarno-Hatta (CGK) Jakarta sebagai titik asal atau tujuan. Ini adalah bukti paling kuat yang mengukuhkan posisi Jakarta sebagai pusat (hub) utama dalam jaringan penerbangan nasional.

2.  **"Golden Triangle" dan Rute Utama (Trunk Routes):**
    * Rute-rute yang menghubungkan tiga kota besar: **Jakarta (CGK), Surabaya (SUB), dan Denpasar (DPS)**, membentuk "segitiga emas" lalu lintas udara di Indonesia. Rute CGK-SUB dan sebaliknya, serta CGK-DPS dan sebaliknya, mendominasi empat dari lima posisi teratas.
    * Selain itu, rute ke **Medan (KNO)** juga menunjukkan volume yang sangat tinggi, menjadikannya salah satu rute utama (trunk route) yang paling vital.

3.  **Keseimbangan Rute Dua Arah (Symmetrical Traffic):**
    * Untuk setiap pasangan kota, volume lalu lintasnya relatif seimbang di kedua arah. Contohnya:
        * CGK-SUB (82,357) vs SUB-CGK (77,992)
        * DPS-CGK (80,126) vs CGK-DPS (77,428)
        * PDG-CGK (50,643) vs CGK-PDG (49,036)
    * Ini menunjukkan aliran penumpang yang stabil dan berkelanjutan, baik untuk perjalanan bisnis, liburan, maupun keluarga di kedua arah, bukan lalu lintas satu arah yang bersifat musiman.
---
#### **Kesimpulan dan Implikasi Strategis**

* **Fokus pada Rute Utama:** Strategi bisnis harus memberikan prioritas tinggi pada rute-rute utama ini (CGK, SUB, DPS, KNO). Ketersediaan, harga yang kompetitif, dan frekuensi penerbangan di rute-rute ini adalah kunci untuk menguasai pangsa pasar terbesar.
* **Validasi Data Regional:** Analisis ini sangat mendukung temuan dari analisis regional (Q3) yang menunjukkan dominasi DKI Jakarta, serta pentingnya Jawa Timur (SUB), Bali (DPS), dan Sumatera Utara (KNO) sebagai pasar sekunder utama.
* **Peluang di Luar Hub:** Sementara rute-rute berbasis di Jakarta adalah yang paling dominan, ini juga membuka pertanyaan strategis: Adakah potensi yang belum tergali di rute-rute non-Jakarta (misalnya, SUB-DPS, SUB-UPG, atau DPS-LOP)? Mengembangkan rute titik-ke-titik (point-to-point) di luar Jakarta bisa menjadi strategi diferensiasi di masa depan.
    

### Q8: What are Top 10 revenue distribution by airline?

In [17]:
# Question 8: Revenue distribution by airline
print("Q8: What are Top 10 revenue distribution by airline?")
revenue_by_airline = df_cleaned.groupby('airline')['harga'].sum().reset_index()
# Sort by average price from highest to lowest and get top 10
revenue_by_airline = revenue_by_airline.sort_values('harga', ascending=False).head(10)
display(revenue_by_airline.style.hide(axis="index"))
fig8 = px.pie(revenue_by_airline, values='harga', names='airline',
               title="Q8: Revenue Distribution by Airline")
fig8.show()

Q8: What are Top 10 revenue distribution by airline?


airline,harga
XG,1828.883574
LE,1095.327377
AZ,761.363028
MI,688.880074
GF,421.415284
HV,258.595013
DB,226.218900
WT,207.528984
QT,121.774009
TS,104.400231


#### **Temuan Utama**

1.  **Pendapatan Tidak Selalu Sejalan dengan Volume:**
    * Fakta paling signifikan adalah bahwa peringkat pendapatan tidak sama persis dengan peringkat volume pesanan (dari Q1). Ini membuktikan bahwa jumlah penumpang yang banyak tidak secara otomatis menjamin pendapatan tertinggi.

2.  **Analisis Kompetitif LE vs. AZ:**
    * Meskipun memiliki jumlah pesanan yang hampir identik di Q1, **Maskapai LE (1095.3)** menghasilkan pendapatan yang jauh lebih tinggi daripada **Maskapai AZ (761.3)**.
    * Ini adalah wawasan kompetitif yang krusial: **Strategi LE lebih efektif dalam menghasilkan uang dari setiap penumpang.** Kemungkinan LE memiliki rata-rata harga tiket yang lebih tinggi, menjual lebih banyak layanan tambahan (ancillaries), atau fokus pada rute yang lebih menguntungkan dibandingkan AZ.

3.  **Dominasi XG Terkonfirmasi:**
    * Maskapai **XG** kokoh di posisi pertama baik dari segi volume maupun pendapatan, dengan total pendapatan **(1828.8)** yang hampir sama dengan gabungan pendapatan peringkat kedua dan ketiga. Ini menegaskan posisinya sebagai pemimpin pasar absolut.

4.  **Pemain Bernilai Tinggi (High-Value Player):**
    * Maskapai **WT** muncul di peringkat ke-8 dalam hal pendapatan **(207.5)**, meskipun tidak masuk dalam 10 besar dari segi volume pesanan. Ini menandakan WT adalah pemain "niche" yang efisien, mampu menghasilkan pendapatan signifikan dari jumlah penumpang yang relatif lebih sedikit, kemungkinan dengan menargetkan segmen premium atau rute bisnis.
---
#### **Kesimpulan dan Implikasi Strategis**

* **Fokus pada "Yield" bukan Hanya Volume:** Daripada hanya mengejar jumlah penumpang, strategi bisnis harus fokus pada peningkatan *yield* (pendapatan per penumpang). Kasus LE vs. AZ adalah contoh sempurna dari pentingnya strategi ini.
* **Pelajaran dari LE:** Perusahaan perlu menganalisis lebih dalam model bisnis LE. Apa yang mereka lakukan secara berbeda? Apakah dari sisi penetapan harga, manajemen rute, atau penawaran produk? Wawasan ini bisa menjadi cetak biru untuk meningkatkan profitabilitas.
* **Identifikasi Peluang Niche:** Keberhasilan pemain seperti WT menunjukkan adanya pasar yang menguntungkan di luar segmen massal. Mengidentifikasi dan melayani segmen premium ini bisa menjadi jalur pertumbuhan baru yang tidak memerlukan persaingan langsung dengan raksasa seperti XG.
    

### Q9: How do booking patterns differ by day of the week?

In [18]:
# Question 9: Booking patterns by day of week
print("Q9: How do booking patterns differ by day of the week?")
df_clean_temp = df_cleaned.copy()
# Check if travel_date is already datetime, if not convert it
if not pd.api.types.is_datetime64_any_dtype(df_clean_temp['travel_date']):
    df_clean_temp['travel_date'] = pd.to_datetime(df_clean_temp['travel_date'])
df_clean_temp['travel_weekday'] = df_clean_temp['travel_date'].dt.day_name()
weekday_bookings = df_clean_temp['travel_weekday'].value_counts().reset_index()
weekday_bookings.columns = ['weekday', 'bookings']
# Reorder by weekday
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_bookings['weekday'] = pd.Categorical(weekday_bookings['weekday'], 
                                           categories=weekday_order, ordered=True)
weekday_bookings = weekday_bookings.sort_values('weekday')
display(weekday_bookings.style.hide(axis="index"))
fig9 = px.bar(weekday_bookings, x='weekday', y='bookings',
               title="Q9: Bookings by Day of Week")
fig9.show()

Q9: How do booking patterns differ by day of the week?


weekday,bookings
Monday,513098
Tuesday,502139
Wednesday,529202
Thursday,523058
Friday,533857
Saturday,505698
Sunday,526872


#### **Temuan Utama**

1.  **Distribusi yang Sangat Merata:**
    * Temuan yang paling mengejutkan adalah betapa meratanya distribusi perjalanan sepanjang minggu. Perbedaan antara hari tersibuk **(Jumat, 533k)** dan hari tersepi **(Selasa, 502k)** hanya sekitar 6.3%.
    * Ini menunjukkan aliran penumpang yang sangat konsisten dan stabil, menandakan adanya perpaduan yang sehat antara **perjalanan bisnis (mid-week)** dan **perjalanan liburan (weekend-adjacent)**.

2.  **Puncak Perjalanan di Hari Jumat dan Minggu:**
    * **Jumat** adalah hari terpopuler untuk melakukan perjalanan, yang sangat sesuai dengan pola orang yang memulai liburan akhir pekan atau pulang ke kampung halaman.
    * **Minggu** adalah hari terpopuler ketiga, yang juga sangat logis karena merupakan hari utama bagi para pelancong untuk kembali ke kota asal mereka sebelum memulai minggu kerja baru.

3.  **Kekuatan Perjalanan Bisnis di Tengah Minggu:**
    * Hari **Rabu** dan **Kamis** menunjukkan volume perjalanan yang sangat tinggi, bahkan sedikit mengungguli hari Minggu. Ini adalah indikator kuat dari segmen perjalanan bisnis yang signifikan, yang sering melakukan perjalanan di pertengahan minggu.

4.  **Kontras dengan Pola Pembuatan Pesanan (Q5):**
    * Analisis ini menunjukkan kontras yang menarik dengan analisis tanggal pembuatan pesanan (Q5), di mana pesanan cenderung memuncak pada hari kerja dan turun drastis di akhir pekan.
    * Ini memperkuat hipotesis: **Pemesanan tiket adalah aktivitas "kerja" yang dilakukan pada hari kerja**, sedangkan **perjalanan itu sendiri terdistribusi lebih merata sepanjang minggu** untuk mengakomodasi baik pelancong bisnis maupun liburan.
---
#### **Kesimpulan dan Implikasi Strategis**

* **Strategi Harga yang Stabil:** Karena permintaan yang relatif merata, tidak ada justifikasi kuat untuk menaikkan harga secara drastis di akhir pekan. Strategi harga bisa dipertahankan agar relatif stabil, dengan kemungkinan penyesuaian kecil pada hari Jumat untuk menangkap permintaan puncak.
* **Peluang Pemasaran Tertarget:** Ada peluang untuk meluncurkan kampanye yang dirancang khusus untuk meningkatkan permintaan pada hari-hari dengan volume sedikit lebih rendah, seperti **Selasa** dan **Sabtu**. Contoh: "Diskon Terbang Selasa" atau "Bonus Liburan Sabtu".
* **Efisiensi Operasional:** Staf bandara dan maskapai dapat merencanakan jadwal mereka dengan asumsi bahwa volume penumpang akan tinggi dan konsisten sepanjang minggu, memungkinkan perencanaan sumber daya yang lebih efisien tanpa siklus "sibuk-sepi" yang ekstrem.
    

## **Ringkasan Eksekutif & Implikasi Strategis (Analisis Q1-Q9)**

**Gambaran Umum**
Analisis komprehensif terhadap data pemesanan penerbangan mengungkap sebuah model bisnis yang sangat terkonsentrasi, didorong oleh volume tinggi pada segmen harga ekonomis. Meskipun menunjukkan pertumbuhan yang kuat pada periode awal, terdapat anomali signifikan yang memerlukan perhatian strategis segera. Laporan ini merangkum tiga pilar strategis utama: struktur pasar, risiko ketergantungan, dan dinamika operasional.

### **1. Struktur Pasar: Model Volume Tinggi, Harga Ekonomis**
Data secara konsisten menunjukkan bahwa mesin penggerak bisnis adalah penjualan tiket dengan volume tinggi pada harga yang sensitif.
* **Dominasi Kelas Ekonomi Promo:** Sebagian besar pemesanan berasal dari kelas tarif terendah (V, X, Q), mengonfirmasi bahwa mayoritas pelanggan memprioritaskan harga.
* **Pemain Utama Berbasis Volume:** Maskapai dengan volume penumpang terbesar (XG, LE, AZ) juga merupakan kontributor pendapatan terbesar, menegaskan model bisnis ini.
* **Jantung Operasi di Jakarta:** Seluruh rute terpopuler terhubung dengan Jakarta (CGK), menjadikannya "super hub" yang tak tergantikan dalam jaringan ini.

### **2. Risiko & Peluang: Ketergantungan vs. Diversifikasi**
Struktur pasar yang terkonsentrasi menciptakan efisiensi, namun juga menyimpan risiko signifikan.
* **Ancaman Ketergantungan Ganda:** Bisnis ini sangat bergantung pada satu maskapai (XG) dan satu hub (CGK). Setiap gangguan pada keduanya akan berdampak masif pada keseluruhan operasional.
* **Studi Kasus Profitabilitas (LE vs. AZ):** Meskipun volume penumpangnya hampir sama, Maskapai LE menghasilkan pendapatan yang jauh lebih tinggi daripada AZ. Ini membuktikan bahwa strategi monetisasi LE lebih efektif dan menjadi pelajaran penting: **fokus harus pada *yield* (pendapatan per penumpang), bukan hanya volume.**
* **Peluang Diversifikasi:** Pasar sekunder utama seperti Surabaya (SUB), Denpasar (DPS), dan Medan (KNO) menunjukkan volume yang kuat. Mengembangkan rute titik-ke-titik (point-to-point) yang tidak melibatkan Jakarta dapat menjadi strategi diferensiasi dan mitigasi risiko.

### **3. Dinamika Operasional & Anomali Kritis**
Pola pemesanan dan perjalanan memberikan wawasan mendalam tentang perilaku pelanggan dan satu anomali besar.
* **Ritme Bisnis Mingguan:** Pemesanan tiket adalah aktivitas hari kerja (Senin-Jumat), sering kali didorong oleh promo. Namun, perjalanan itu sendiri terdistribusi secara merata sepanjang minggu. Ini memisahkan "aksi jual" (hari kerja) dari "aksi bepergian" (sepanjang minggu).
* **Musim Puncak Teridentifikasi:** Permintaan perjalanan memuncak selama periode libur Lebaran dan Natal-Tahun Baru, memberikan pola yang dapat diprediksi untuk perencanaan.
* **INVESTIGASI KRITIS - Penurunan Drastis Awal 2019:** Data menunjukkan volume bisnis, baik pemesanan maupun perjalanan, jatuh secara drastis setelah Januari 2019. Ini adalah **temuan paling krusial dan berisiko tinggi**. Tanpa memahami penyebabnya (perubahan bisnis, faktor eksternal, atau kesalahan data), semua perencanaan strategis dan peramalan menjadi tidak valid.

### **Implikasi & Rekomendasi Strategis Utama**
1.  **Prioritas #1: Investigasi Anomali 2019.** Selidiki segera penyebab penurunan volume setelah Januari 2019. Jawaban atas pertanyaan ini akan menentukan arah strategi ke depan.
2.  **Geser Fokus dari Volume ke Nilai (Yield).** Pelajari model bisnis Maskapai LE. Implementasikan strategi untuk meningkatkan pendapatan per penumpang, misalnya melalui layanan tambahan (ancillaries) atau manajemen harga yang lebih dinamis.
3.  **Buat Rencana Diversifikasi Rute.** Kurangi ketergantungan pada Jakarta dengan secara aktif mengembangkan rute antar-hub sekunder (misal: SUB-UPG, DPS-LOP) untuk membangun keunggulan kompetitif baru.
4.  **Optimalkan Pemasaran & Operasional.** Luncurkan kampanye promosi besar pada hari kerja. Pertahankan staf operasional pada tingkat yang konsisten sepanjang minggu, dengan antisipasi puncak pada hari Jumat.
    